In [2]:
from __future__ import annotations
import sys
import os
import tqdm
import einops
import tabulate
import gc
import torch
from pathlib import Path
from dotenv import load_dotenv
sys.path.append(Path.cwd().parent.as_posix())
sys.path = list(set(sys.path))
from typing import Optional, List, Dict
from transformers import LlamaTokenizer, LlamaForCausalLM
import dotenv
import os
from pydantic import BaseModel, ConfigDict
from datasets import load_dataset, concatenate_datasets, Dataset # XXX why the FUCK does this break?
dotenv.load_dotenv()
sys.path

['',
 '/mnt/align3_drive/adrianoh/miniconda3/envs/ifyoudont/lib/python3.12/site-packages',
 '/mnt/align3_drive/adrianoh/miniconda3/envs/ifyoudont/lib/python312.zip',
 '/mnt/align3_drive/adrianoh/miniconda3/envs/ifyoudont/lib/python3.12/lib-dynload',
 '/mnt/align3_drive/adrianoh/miniconda3/envs/ifyoudont/lib/python3.12',
 '/mnt/align3_drive/adrianoh/miniconda3/envs/ifyoudont/lib/python3.12/site-packages/setuptools/_vendor',
 '/mnt/align3_drive/adrianoh/git/ApartModelScoping']

In [15]:
import sys
sys.path
max_dataset_size = 10_000
import zipnn
from datasets import load_dataset

#### KNOWLEDGE QA DATASETS ####
# TODO(Adriano) canonicalize the keys to make this easier for preprocessing and the like
class Datasets(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)
    datasets: List[Dataset] # NOTE: these are modified to have both labels added
    label_str2int: Dict[str, int]
    label_int2str: Dict[int, str]

dataset_names = ["camel-ai/physics", "camel-ai/biology", "camel-ai/chemistry", "camel-ai/math"]
dataset_sizes = [max_dataset_size, max_dataset_size, max_dataset_size, max_dataset_size]

def insert_label(x: Dict, label: int) -> Dict:
    x.update({"label": label})
    return x

def load_datasets(dataset_names: List[str], dataset_sizes: List[int]) -> Tuple[List[Dataset], Dict[str, int], Dict[int, str]]:
    datasets = [
        load_dataset(dataset_name, split="train").shuffle(seed=42).select(range(dataset_size)) 
        for dataset_name, dataset_size in tqdm.tqdm(
            zip(dataset_names, dataset_sizes),
            desc="Loading datasets",
            total=len(dataset_names)
        )
    ]
    label_str2int = {dataset_name: i for i, dataset_name in enumerate(dataset_names)}
    label_int2str = {v: k for k, v in label_str2int.items()}
    datasets_with_labels = [
        dataset.map(lambda x: insert_label(x, label_str2int[dataset_name])) for dataset, dataset_name in zip(datasets, dataset_names)
    ]
    return datasets_with_labels, label_str2int, label_int2str

datasets, label_str2int, label_int2str = load_datasets(dataset_names, dataset_sizes)

# TODO(Adriano) get these working so we an have full diversity plz
# code_camelai = load_dataset("camel-ai/exit
# code", split="train").shuffle(seed=42).select(range(max_dataset_size))
# ai_society_camelai = load_dataset("camel-ai/ai_society", split="train").shuffle(seed=42).select(range(max_dataset_size))

Loading datasets: 100%|██████████| 4/4 [00:04<00:00,  1.24s/it]


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [8]:
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id="camel-ai/ai_society", repo_type="dataset", filename="ai_society_chat.tar.gz",
                local_dir="datasets/", local_dir_use_symlinks=False)

/mnt/align3_drive/adrianoh/miniconda3/envs/ifyoudont/lib/python3.12/site-packages/huggingface_hub/file_download.py:1204: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


ai_society_chat.tar.gz:   0%|          | 0.00/30.4M [00:00<?, ?B/s]

'datasets/ai_society_chat.tar.gz'

In [12]:
# code_camelai = load_dataset("camel-ai/code", split="train")
# dataset = load_dataset("camel-ai/code", name="default", split="train")